### ELMDA (Embeddings from Language Models)
- Bi LSTM 기반 양방향 언어모델 구조
    - 언어모델 : 이전 단어들을 보고 다음 단어를 예측하는 모델
    - Bi - LSTM : 순방향 + 역방향
    - ELMO : Bi-LSTM 을 여러층 쌓아올려 특정 단어의 의미를 파악할때 그 단어의 앞 문맥과 뒤 문맥을 동시에 고려
- 문맥에따라 변하는 dynamic 임베딩 원리 :
    - 1교시 static 임베딩은 사전에서 단어 찾는거와 유사
    - 임베딩 값이 다르게 생성되어서 동음이의어문제를 완벽하게 해결
- feature -based 전이학습(Transfer learning)
    - 전이 학습 : 거대 데이터로 미리 학습된 지식을 가져와서 내 문제에 맞춰 사용하는 기법
    - feature - based : 기존에 쓰던 모델(사용자의 모델) 의 내부 구조를 바꾸지 않음. ELMO 가 만들어준 똑똑한
    동적 벡터를 단순히 이어 붙여서 특징으로 추가해주는 방식

In [46]:
# Bi - LSTTM
import torch
import torch.nn as nn
import torch.nn.functional as F

embed_size = 10
hidden_size = 12

# input size : 입력되는 벡터의 차원
# hidden_size : LSTM 내부적으로 사용하는 은닉상태
# num_layers : LSTM 레이어의 수
# bidirectional : 양방향 LSTM 여부
lstm = nn.LSTM(input_size=embed_size, hidden_size=hidden_size, num_layers=1, bidirectional=True)

# 가상의 데이터 입력 (문장 1개 , 5개의단어 , 각 단어는 10차원)
dmy_input = torch.rand(1,5,embed_size) # (batch_size, seq_length, embed_size)
output, (h_n, c_n) = lstm(dmy_input)

dmy_input.shape, output.shape, h_n.shape, c_n.shape
# 정방향 + 역방향 -> 24차원

(torch.Size([1, 5, 10]),
 torch.Size([1, 5, 24]),
 torch.Size([2, 5, 12]),
 torch.Size([2, 5, 12]))

In [ ]:
# bank의 벡터값이 문맥에 따라서 다르게 나오는지 코사인 유사도로 확인
# F.cosine_similarity(텐서1, 텐서2, dim) : dim 에 해당하는 차원을 기준으로 계산   -1 ~ 1
# 임시 단어사전 구성 10차원 정적 임베딩

word_to_idx = {'I':0, 'deposited':1, 'money':2, 'in':3, 'the':4, 'bank':5, 'walked':6, 'by':7, 'river':8, '.':9}
static_embedding = nn.Embedding(num_embeddings=10, embedding_dim=embed_size)

# 문장 A: I deposited money in the bank .
sentence_A_idx = torch.tensor([[word_to_idx[w] for w in ['I', 'deposited', 'money', 'in', 'the', 'bank', '.']]])
# 문장 B: I walked by the river bank .
sentence_B_idx = torch.tensor([[word_to_idx[w] for w in ['I', 'walked', 'by', 'the', 'river', 'bank', '.']]])

emb_a = static_embedding(sentence_A_idx) # (1, 7, 10)
emb_b = static_embedding(sentence_B_idx) # (1, 7, 10)

# 2 . Bi - LSTM
elmo_out_A , _ = lstm(emb_a)
elmo_out_B , _ = lstm(emb_b)

# 3. 두 문장에서 bank 단어의 위치(인덱스) 둘다 5
# 정적 임베딩에서 bank 벡터간 유사도(A문장의 bank vs B문장의 bank)
static_bank_A = emb_a[0,5,:] # (10,)
static_bank_B = emb_b[0,5,:] # (10,)
static_similarity = F.cosine_similarity(static_bank_A, static_bank_B, dim=0)

print(static_similarity.item())
# ELMO : 문맥에 따른 단어의 의미를 반영한 임베딩
elmo_bank_A = elmo_out_A[0,5,:] # (24,)
elmo_bank_B = elmo_out_B[0,5,:] # (24,)

elmo_similarity = F.cosine_similarity(elmo_bank_A.unsqueeze(0), elmo_bank_B.unsqueeze(0))
print(elmo_similarity.item())

0.9999998807907104
1.0000001192092896


In [48]:
# feature-based 전이학습
# ELMO가 생성한 문맥을 반영한 벡터를 기존 딥러닝 분류기에 어떻게 전이학습 가져다 쓰는지 Concat 연산을 통해 확인
# 기존 10차원이고 ELMO가 24차원이므로 34차원으로 만들어서 기존 모델에 입력으로 넣어주는 방식